# PeriClock feature interpretation and expression trends

Paper: Fig. 4a–b and expression-trend analysis underlying Extended Data Fig. 3. Inputs: the separately downloaded original annotated atlas, plus the bundled 173-row stable-feature/coefficient table and 169-row expression-only age-trend matrix. Nonzero-feature interpretation and expression-trend plotting use their original respective filters; neither changes the 173-input predictor. Environment: Python/Scanpy, NumPy, pandas, SciPy, Matplotlib, seaborn and GSEApy; online Enrichr resources are additionally required. Outputs: age-trend heatmap, subtype feature-count PDF, signed enrichment CSVs and KEGG pathway–gene heatmap under `OUTPUT_DIR/fig4`. No previous tutorial variables are required. Validation: the expression-trend section was executed sequentially with the bundled CSVs and produced a valid PDF; the full notebook and atlas/enrichment execution remain unverified.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Author-confirmed integrated atlas `inte_final.h5ad`, bundled coefficient/trend tables, named Enrichr libraries and licensed Arial fonts; [integrated atlas](../RESOURCE_DOWNLOAD.md#2-integrated-pbmc-atlas) and [other references](../RESOURCE_DOWNLOAD.md#8-other-reference-databases). Historical `bbknn_all.h5ad` is byte-identical to this atlas, verified by full SHA256.

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
ANNOTATED_ATLAS = Path(os.environ.get("PERICLOCK_ANNOTATED_ATLAS", str(RESOURCE_DIR / "datasets/inte_final.h5ad"))).expanduser().resolve()
FEATURE_COEFFICIENTS = Path(os.environ.get("PERICLOCK_FEATURE_COEFFICIENTS", str(PROJECT_ROOT / "resources/features/stable_feature_coef.csv"))).expanduser().resolve()
EXPRESSION_TRENDS = Path(os.environ.get("PERICLOCK_EXPRESSION_TRENDS", str(PROJECT_ROOT / "resources/features/features_trend_trainset.csv"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/04_stable_feature_interpretation"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [ANNOTATED_ATLAS, FEATURE_COEFFICIENTS, EXPRESSION_TRENDS]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Expression trends across age groups

Source: cells 0 and 2; analysis underlying the Extended Data Fig. 3 expression-trend panel. Age trends concern **gene-expression features only**, not the cell-proportion features. The selected input table contains 169 expression features and four proportions (173 total); the fitted model has 168 nonzero expression coefficients plus four nonzero proportion coefficients (172 total). The original trend matrix contains all 169 selected expression features, including the zero-coefficient `CD4Tcm_CD27_SESN3`. This original plotting behavior is retained: no proportion rows are added and no expression row is removed to match a caption count. The table has eight age-group columns A–H; row z-scoring, slope ordering and coefficient annotations remain unchanged.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scanpy as sc

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from matplotlib.colors import TwoSlopeNorm

# ===============================
# 0. 参数与数据
# ===============================
coef_file = str(FEATURE_COEFFICIENTS)
heat_file = str(EXPRESSION_TRENDS)
output_pdf = str(OUTPUT_DIR / "fig4/stable_feature_age_heatmap.pdf")

heat_df = pd.read_csv(heat_file, index_col=0)
df_coef = pd.read_csv(coef_file)

# ===============================
# 1. 每个 feature 按年龄组做 row z-score
# ===============================
heat_df_scaled = heat_df.apply(lambda row: stats.zscore(row, nan_policy="omit"), axis=1, result_type="broadcast")
heat_df_scaled = heat_df_scaled.replace([np.inf, -np.inf], np.nan).dropna(how="all")

# 去掉完全没有变化的 feature
heat_df_scaled = heat_df_scaled.loc[heat_df_scaled.std(axis=1) > 0]

# ===============================
# 2. 按年龄趋势 slope 排序
# ===============================
x = np.arange(heat_df_scaled.shape[1])

def calculate_slope(row):
    valid = np.isfinite(row.values)
    if valid.sum() < 2:
        return np.nan
    return np.polyfit(x[valid], row.values[valid], 1)[0]

slopes = heat_df_scaled.apply(calculate_slope, axis=1).dropna()
heat_df_sorted = heat_df_scaled.loc[slopes.sort_values(ascending=False).index]

# ===============================
# 3. 对齐模型 coefficient
# ===============================
coef_series = df_coef.drop_duplicates("Feature").set_index("Feature")["Coefficient"]
coef_aligned = coef_series.reindex(heat_df_sorted.index)

# ===============================
# 4. 作图参数
# ===============================
n_features = len(heat_df_sorted)
fig_height = max(5, 0.24 * n_features)
fig_width = 8

sns.set_theme(style="white")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# 主热图 + colorbar + coefficient
fig = plt.figure(figsize=(fig_width, fig_height))
gs = fig.add_gridspec(1, 3, width_ratios=[10, 0.18, 1.0], wspace=0.35)

ax_heat = fig.add_subplot(gs[0, 0])
ax_cbar = fig.add_subplot(gs[0, 1])
ax_coef = fig.add_subplot(gs[0, 2])

# ===============================
# 5. 主表达热图
# ===============================
sns.heatmap(
    heat_df_sorted,
    ax=ax_heat,
    cmap="RdBu_r",
    center=0,
    vmin=-2.5,
    vmax=2.5,
    linewidths=0.15,
    linecolor="white",
    yticklabels=True,
    cbar_ax=ax_cbar,
    cbar_kws={"label": "Row z-score"}
)

ax_heat.set_title("Stable feature expression across age groups", fontsize=12, fontweight="bold", pad=12)
ax_heat.set_xlabel("Age group", fontsize=10)
ax_heat.set_ylabel("Stable features", fontsize=10)
ax_heat.tick_params(axis="x", labelsize=9, rotation=0)
ax_heat.tick_params(axis="y", labelsize=7 if n_features > 50 else 8, length=0)

# colorbar
ax_cbar.tick_params(axis="y", labelsize=8, length=2)
ax_cbar.set_ylabel("Row z-score", fontsize=9, labelpad=4)

# ===============================
# 6. 右侧 coefficient annotation
# ===============================
coef_plot = coef_aligned.to_numpy().reshape(-1, 1)
coef_abs_max = np.nanmax(np.abs(coef_plot))

sns.heatmap(
    coef_plot,
    ax=ax_coef,
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-coef_abs_max, vcenter=0, vmax=coef_abs_max),
    yticklabels=False,
    xticklabels=["Coefficient"],
    cbar=False,
    linewidths=0.15,
    linecolor="white"
)

ax_coef.tick_params(axis="x", labelsize=8, rotation=90, length=0)
ax_coef.set_xlabel("")
ax_coef.set_ylabel("")
ax_coef.set_title("Model", fontsize=9, pad=12)

# coefficient 格子中标数值
for i, coef in enumerate(coef_aligned.values):
    if pd.notna(coef):
        ax_coef.text(0.5, i + 0.5, f"{coef:.3f}", ha="center", va="center", fontsize=6)

# ===============================
# 7. 保存
# ===============================
os.makedirs(os.path.dirname(output_pdf), exist_ok=True)
plt.savefig(output_pdf, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

print(f"Saved to: {output_pdf}")

## Nonzero features across cell subtypes (Fig. 4a)

Source: cells 5–6, 8 and 12. The atlas supplies the original subtype-to-lineage annotations (`cell_subtype_final`, `cell_subtype`); the coefficient table supplies `Feature`, `Coefficient`, `Frequency`, `Celltype`, `Gene` and `FeatureType`. The original explicit zero-coefficient filter is retained. Loading the full atlas can require substantial memory; this section was not executed. No cohort expression processing is replaced by a surrogate table.

In [ ]:
adata = sc.read_h5ad(str(ANNOTATED_ATLAS))
adata

In [ ]:
stable_fea = pd.read_csv(str(FEATURE_COEFFICIENTS))
stable_fea

In [ ]:
# 把这个coefficient为0的特征去掉
stable_fea = stable_fea.loc[
    stable_fea["Coefficient"] != 0
].copy()

In [ ]:
Path("fig4").mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
from matplotlib.patches import Patch


# ============================================================
# 2. 整理stable_fea中的细胞亚型
# ============================================================
stable_plot = stable_fea.copy()

# 优先使用已有的Celltype列
if "Celltype" in stable_plot.columns:
    stable_plot["Cell_Subtype"] = stable_plot["Celltype"].astype(str)

# 如果没有Celltype列，则通过Feature去掉末尾基因名获得亚型
elif {"Feature", "Gene"}.issubset(stable_plot.columns):
    stable_plot["Cell_Subtype"] = stable_plot.apply(
        lambda row: row["Feature"][
            :-(len(str(row["Gene"])) + 1)
        ] if str(row["Feature"]).endswith(
            "_" + str(row["Gene"])
        ) else np.nan,
        axis=1
    )

else:
    raise ValueError(
        "stable_fea中必须包含Celltype列，"
        "或者同时包含Feature和Gene列。"
    )


# ============================================================
# 3. 从adata中建立“细胞亚型 -> 细胞大类”映射
# ============================================================
annotation_df = (
    adata.obs[
        ["cell_subtype_final", "cell_subtype"]
    ]
    .dropna()
    .astype(str)
    .copy()
)

# 检查一个亚型是否被标注到多个大类
mapping_check = (
    annotation_df
    .groupby("cell_subtype_final")["cell_subtype"]
    .nunique()
)

conflicting_subtypes = mapping_check[mapping_check > 1]

if len(conflicting_subtypes) > 0:
    print("警告：以下亚型对应多个细胞大类，将使用出现次数最多的大类：")
    print(conflicting_subtypes)

# 对每个亚型使用出现次数最多的大类
subtype_to_major = (
    annotation_df
    .groupby("cell_subtype_final")["cell_subtype"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

stable_plot["Major_Celltype"] = (
    stable_plot["Cell_Subtype"]
    .map(subtype_to_major)
)

# 检查是否有亚型没有匹配到大类
unmatched_subtypes = sorted(
    stable_plot.loc[
        stable_plot["Major_Celltype"].isna(),
        "Cell_Subtype"
    ].unique()
)

if len(unmatched_subtypes) > 0:
    print("\n以下stable_fea亚型没有在adata中匹配到：")
    print(unmatched_subtypes)

    # 未匹配项标记为Unmatched，而不是直接删除
    stable_plot["Major_Celltype"] = (
        stable_plot["Major_Celltype"]
        .fillna("Unmatched")
    )


# ============================================================
# 1. 加载 Arial 字体
# ============================================================
font_dir = str(FONT_DIR)

for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 14,
    "font.weight": "bold",
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 13,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.3,
    "legend.fontsize": 13,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ============================================================
# 2. 统计
# ============================================================
plot_df = (
    stable_plot
    .groupby(["Cell_Subtype", "Major_Celltype"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

# ============================================================
# 3. 低饱和暗色系
# 蓝 / 粉 / 橙 / 紫 / 绿 / 青 / 棕
# ============================================================
dark_palette = [
    "#5277A8",   # muted blue
    "#B26C7C",   # muted rose pink
    "#C07A45",   # muted orange
    "#80699B",   # muted purple
    "#668B62",   # muted green
    "#4F8C8A",   # muted teal
    "#A38163",   # muted brown
    "#7C7C7C"    # muted gray
]

major_types = plot_df["Major_Celltype"].drop_duplicates().tolist()

major_color_dict = {
    celltype: dark_palette[i % len(dark_palette)]
    for i, celltype in enumerate(major_types)
}

plot_df["Color"] = plot_df["Major_Celltype"].map(major_color_dict)

# ============================================================
# 4. 作图
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.barh(
    plot_df["Cell_Subtype"],
    plot_df["Count"],
    color=plot_df["Color"],
    edgecolor="#444444",
    linewidth=0.6,
    height=0.72
)

ax.invert_yaxis()

# ============================================================
# 5. 数值
# ============================================================
xmax = plot_df["Count"].max()

for bar in bars:
    width = bar.get_width()
    ax.text(
        width + xmax * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{int(width)}",
        va="center",
        ha="left",
        fontsize=13,
        fontweight="bold",
        fontfamily="Arial"
    )

ax.set_xlim(0, xmax * 1.12)

# ============================================================
# 6. 坐标轴
# ============================================================
ax.set_xlabel(
    "Number of Stable Features",
    fontsize=16,
    fontweight="bold"
)

ax.set_ylabel("")

ax.set_title(
    "Stable Aging Features across Cell Subtypes",
    fontsize=18,
    fontweight="bold",
    pad=12
)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontfamily("Arial")
    label.set_fontweight("bold")

# ============================================================
# 7. 网格
# ============================================================
ax.set_axisbelow(True)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.22,
    color="#777777"
)

sns.despine(ax=ax)

# ============================================================
# 8. 图例
# ============================================================
handles = [
    Patch(
        facecolor=major_color_dict[m],
        edgecolor="#444444",
        linewidth=0.5,
        label=m
    )
    for m in major_types
]

legend = ax.legend(
    handles=handles,
    title="Major cell type",
    frameon=False,
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=13,
    title_fontsize=14
)

plt.setp(legend.get_title(), fontfamily="Arial", fontweight="bold")
plt.setp(legend.get_texts(), fontfamily="Arial", fontweight="bold")

# ============================================================
# 9. 保存
# ============================================================
plt.tight_layout()

plt.savefig(
    "fig4/stable_feature_celltype_barplot.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Signed enrichment and pathway–gene weights (Fig. 4b)

Source: cells 15–16 and 21–22. GSEApy queries the original named Enrichr libraries, then the original code builds the KEGG heatmap from the tables it just wrote. Network access and the specified library versions are required. The original function catches enrichment exceptions: inspect the messages and confirm both signed tables were produced. This preparation did not execute enrichment or verify the live service. Proteomics, HAGR, TF/LISA and longevity panels are not reproduced by this notebook.

In [ ]:
import os
import numpy as np
import pandas as pd

# 仅保留表达特征
df = stable_fea.loc[stable_fea["FeatureType"] == "Expression"].copy()

# 确保Gene和Coefficient有效
df = df.dropna(subset=["Gene", "Coefficient"])

df["Gene"] = (df["Gene"].astype(str).str.strip().str.upper())

# 正系数特征：表达增加时模型预测年龄增加
up_genes = df.loc[df["Coefficient"] > 0,["Gene", "Celltype", "Coefficient", "Frequency", "Feature"]].copy()

# 负系数特征：表达增加时模型预测年龄降低
down_genes = df.loc[df["Coefficient"] < 0,["Gene", "Celltype", "Coefficient", "Frequency", "Feature"]].copy()

# 富集分析要求gene symbol列表，去除重复基因
up_gene_list = sorted(up_genes["Gene"].dropna().unique().tolist())

down_gene_list = sorted(down_genes["Gene"].dropna().unique().tolist())

print(f"Positive-coefficient features: {len(up_genes)}")
print(f"Unique positive-coefficient genes: {len(up_gene_list)}")

print(f"Negative-coefficient features: {len(down_genes)}")
print(f"Unique negative-coefficient genes: {len(down_gene_list)}")

In [ ]:
import gseapy as gp

save_dir = "fig4/stable_feature_enrichment"
os.makedirs(save_dir, exist_ok=True)

gene_sets = {
    "GO_BP": "GO_Biological_Process_2025",
    "GO_CC": "GO_Cellular_Component_2025",
    "GO_MF": "GO_Molecular_Function_2025",
    "KEGG": "KEGG_2021_Human"
}


def run_enrichr(
    gene_list,
    group_name,
    gene_sets,
    outdir,
    organism="Human"
):
    """
    使用Enrichr对一个gene list执行GO和KEGG富集分析。

    Parameters
    ----------
    gene_list : list
        去重后的基因symbol列表。
    group_name : str
        分组名称，例如Positive或Negative。
    gene_sets : dict
        数据库简称和Enrichr数据库名称。
    outdir : str
        输出目录。
    organism : str
        物种，默认为Human。

    Returns
    -------
    result_dict : dict
        每个数据库对应一个富集结果DataFrame。
    """
    if len(gene_list) < 2:
        raise ValueError(
            f"{group_name}基因数少于2，不能进行可靠的富集分析。"
        )

    group_dir = os.path.join(outdir, group_name)
    os.makedirs(group_dir, exist_ok=True)

    result_dict = {}

    for database_short, database_name in gene_sets.items():

        print(
            f"Running {group_name}: "
            f"{database_short} ({database_name})"
        )

        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=database_name,
                organism=organism,
                outdir=None,
                cutoff=1.0,
                no_plot=True
            )

            result = enr.results.copy()

            if result.empty:
                print(
                    f"  No enrichment results: "
                    f"{group_name} - {database_short}"
                )
                result_dict[database_short] = pd.DataFrame()
                continue

            # 统一数据库名称
            result["Database"] = database_short
            result["Group"] = group_name

            # 计算绘图需要的指标
            result["minus_log10_FDR"] = -np.log10(
                result["Adjusted P-value"].clip(lower=1e-300)
            )

            # Overlap通常是“命中基因数/通路基因数”
            overlap_split = (
                result["Overlap"]
                .astype(str)
                .str.split("/", expand=True)
            )

            result["Overlap_Count"] = pd.to_numeric(
                overlap_split[0],
                errors="coerce"
            )

            result["Pathway_Size"] = pd.to_numeric(
                overlap_split[1],
                errors="coerce"
            )

            result["Gene_Ratio"] = (
                result["Overlap_Count"]
                / result["Pathway_Size"]
            )

            result = result.sort_values(
                ["Adjusted P-value", "Combined Score"],
                ascending=[True, False]
            )

            result.to_csv(
                os.path.join(
                    group_dir,
                    f"{group_name}_{database_short}_enrichment.csv"
                ),
                index=False
            )

            result_dict[database_short] = result

        except Exception as e:
            print(
                f"  Failed: {group_name} - "
                f"{database_short}: {e}"
            )
            result_dict[database_short] = pd.DataFrame()

    return result_dict

up_enrichment = run_enrichr(
    gene_list=up_gene_list,
    group_name="Positive",
    gene_sets=gene_sets,
    outdir=save_dir
)

down_enrichment = run_enrichr(
    gene_list=down_gene_list,
    group_name="Negative",
    gene_sets=gene_sets,
    outdir=save_dir
)

In [ ]:
stable_fea = pd.read_csv(str(FEATURE_COEFFICIENTS))
stable_fea

In [ ]:
import os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl, matplotlib.font_manager as fm, seaborn as sns
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

# ============================================================
# 1. 路径与参数
# ============================================================
base_dir = "fig4/stable_feature_enrichment"
positive_file = os.path.join(base_dir, "Positive", "Positive_KEGG_enrichment.csv")
negative_file = os.path.join(base_dir, "Negative", "Negative_KEGG_enrichment.csv")
save_dir = os.path.join(base_dir, "figures")
output_pdf = os.path.join(save_dir, "KEGG_pathway_gene_heatmap.pdf")
os.makedirs(save_dir, exist_ok=True)

fdr_cutoff = 0.05
top_positive = 8
top_negative = 10

# ============================================================
# 2. Arial
# ============================================================
font_dir = str(FONT_DIR)
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 13, "font.weight": "bold",
    "axes.labelsize": 14, "axes.labelweight": "bold", "xtick.labelsize": 11, "ytick.labelsize": 11,
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ============================================================
# 3. 读取 KEGG 结果
# ============================================================
positive_raw = pd.read_csv(positive_file)
negative_raw = pd.read_csv(negative_file)

def clean_term_name(term):
    term = str(term).replace("_", " ")
    return re.sub(r"\s*\(GO:\d+\)$", "", term)

def prepare_enrichment_result(df, top_n, fdr_cutoff=0.05):
    df = df.loc[df["Adjusted P-value"] < fdr_cutoff].copy()
    if df.empty: return df
    df = df.sort_values(["Adjusted P-value", "Combined Score"], ascending=[True, False]).head(top_n).copy()
    df["Term_clean"] = df["Term"].map(clean_term_name)
    return df

positive_kegg = prepare_enrichment_result(positive_raw, top_positive, fdr_cutoff)
negative_kegg = prepare_enrichment_result(negative_raw, top_negative, fdr_cutoff)
positive_kegg["Direction"] = "Positive"
negative_kegg["Direction"] = "Negative"
kegg_plot = pd.concat([positive_kegg, negative_kegg], ignore_index=True)

if kegg_plot.empty: raise ValueError("没有 FDR < 0.05 的 KEGG 富集结果。")

# ============================================================
# 4. 拆分通路基因
# ============================================================
def split_gene_string(x):
    if pd.isna(x): return []
    return [g.strip().upper() for g in re.split(r"[;,/]", str(x)) if g.strip()]

kegg_plot["Gene_list"] = kegg_plot["Genes"].map(split_gene_string)

# ============================================================
# 5. stable gene score
# 这里默认 stable_fea 已经在环境中
# ============================================================
stable_expr = stable_fea.loc[(stable_fea["FeatureType"] == "Expression") & stable_fea["Gene"].notna() & stable_fea["Coefficient"].notna() & stable_fea["Frequency"].notna()].copy()
stable_expr["Gene"] = stable_expr["Gene"].astype(str).str.strip().str.upper()
stable_expr["Coefficient"] = pd.to_numeric(stable_expr["Coefficient"], errors="coerce")
stable_expr["Frequency"] = pd.to_numeric(stable_expr["Frequency"], errors="coerce")
stable_expr = stable_expr.dropna(subset=["Coefficient", "Frequency"])
stable_expr = stable_expr.loc[stable_expr["Coefficient"].abs() > 1e-10].copy()
stable_expr["Weighted_coefficient"] = stable_expr["Coefficient"] * stable_expr["Frequency"]
gene_score = stable_expr.groupby("Gene")["Weighted_coefficient"].sum()

# ============================================================
# 6. 构建 Pathway × Gene 矩阵
# ============================================================
all_pathway_genes = sorted({gene for gene_list in kegg_plot["Gene_list"] for gene in gene_list})
matrix = pd.DataFrame(np.nan, index=kegg_plot["Term_clean"], columns=all_pathway_genes, dtype=float)

for _, row in kegg_plot.iterrows():
    for gene in row["Gene_list"]:
        matrix.loc[row["Term_clean"], gene] = gene_score.get(gene, 0)

# ============================================================
# 7. 基因筛选
# ============================================================
gene_occurrence = matrix.notna().sum(axis=0)
gene_abs_score = pd.Series({gene: abs(gene_score.get(gene, 0)) for gene in matrix.columns})
shared_genes = gene_occurrence[gene_occurrence >= 2].index.tolist()
top_effect_genes = gene_abs_score.sort_values(ascending=False).head(15).index.tolist()
selected_genes = [g for g in dict.fromkeys(shared_genes + top_effect_genes) if g in matrix.columns]

matrix_plot = matrix[selected_genes].copy()
row_order = positive_kegg["Term_clean"].tolist() + negative_kegg["Term_clean"].tolist()
matrix_plot = matrix_plot.reindex(row_order)

# ============================================================
# 8. gene clustering
# ============================================================
gene_distance = pdist(matrix_plot.fillna(0).T, metric="correlation")
gene_distance = np.nan_to_num(gene_distance, nan=0.0, posinf=0.0, neginf=0.0)
gene_linkage = linkage(gene_distance, method="average")

# ============================================================
# 9. 颜色：稍暗的蓝 - 白 - 红
# ============================================================
heatmap_cmap = LinearSegmentedColormap.from_list(
    "paper_RdBu", ["#3D73AD", "#F7F7F7", "#BF4545"], N=256
)

max_abs = np.nanmax(np.abs(matrix_plot.values))
if max_abs == 0: max_abs = 1
norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)

# ============================================================
# 10. clustermap
# ============================================================
g = sns.clustermap(
    matrix_plot,
    row_cluster=False,
    col_cluster=True,
    col_linkage=gene_linkage,
    cmap=heatmap_cmap,
    norm=norm,
    mask=matrix_plot.isna(),
    linewidths=0.4,
    linecolor="white",
    figsize=(7.2, 6.6),
    dendrogram_ratio=(0.02, 0.12),
    cbar_pos=None
)

heatmap_ax = g.ax_heatmap
dendro_ax = g.ax_col_dendrogram

# ============================================================
# 11. 布局
# 保持你原始代码的整体结构，只给gene label留足空间
# ============================================================
heatmap_ax.set_position([0.18, 0.11, 0.56, 0.53])
dendro_ax.set_position([0.18, 0.755, 0.56, 0.075])

# ============================================================
# 12. x轴 gene labels
# ============================================================
heatmap_ax.xaxis.tick_top()

heatmap_ax.set_xticklabels(
    heatmap_ax.get_xticklabels(),
    rotation=90,
    ha="center",
    va="bottom",
    fontsize=9.5,
    fontweight="bold",
    fontfamily="Arial"
)

heatmap_ax.tick_params(axis="x", pad=3, length=0)

# ============================================================
# 13. y轴 pathway labels
# ============================================================
heatmap_ax.set_yticklabels(
    matrix_plot.index,
    rotation=0,
    fontsize=11,
    fontweight="bold",
    fontfamily="Arial"
)

heatmap_ax.tick_params(axis="y", length=0, pad=6)
heatmap_ax.set_xlabel("")
heatmap_ax.set_ylabel("")

# ============================================================
# 14. Positive / Negative separator
# ============================================================
n_positive = len(positive_kegg)

if n_positive > 0 and len(negative_kegg) > 0:
    heatmap_ax.axhline(n_positive, color="#333333", linewidth=1.2)

# ============================================================
# 15. colorbar
# ============================================================
cbar_ax = g.fig.add_axes([0.045, 0.27, 0.020, 0.32])

cb = plt.colorbar(heatmap_ax.collections[0], cax=cbar_ax)

cb.set_label(
    "Gene score\nΣ(Coefficient × Frequency)",
    fontsize=10,
    fontweight="bold",
    fontfamily="Arial",
    labelpad=7
)

cb.ax.tick_params(labelsize=9, width=1)

for label in cb.ax.get_yticklabels():
    label.set_fontfamily("Arial")
    label.set_fontweight("bold")

# ============================================================
# 16. borders
# ============================================================
for spine in heatmap_ax.spines.values(): spine.set_visible(False)
for spine in dendro_ax.spines.values(): spine.set_visible(False)

# ============================================================
# 17. save
# ============================================================
plt.savefig(output_pdf, dpi=300, bbox_inches="tight", pad_inches=0.05)
plt.show()

print(f"Saved to: {output_pdf}")